# KDIC 답변 시스템 A안 — Raw Top-5 → HCX-005

이 노트북은 **검색 성능 비교가 끝났다고 가정한 답변 생성 A안 기준선**입니다.

전체 흐름은 다음과 같습니다.

```text
사용자 질문
  → Dense-structured-v2 상위 20개
  → BM25 + Nori-none 상위 20개
  → Weighted RRF(0.7 / 0.3, rrf_k=10)
  → 최종 Top 5 원본 청크
  → HCX-005에 원본 청크를 직접 전달
  → 쉬운 답변
  → [자세한 답변 보기] 클릭 시 같은 Top 5로 전문가 답변 생성
```

고정 검색 조건:

- Dense: HCX `bge-m3`, `title + section_title + content` 구조화 입력 v2
- Sparse: Elasticsearch BM25 + 공식 `analysis-nori`, `decompound_mode=none`
- 결합: Dense 0.7 / BM25 0.3 Weighted RRF
- `rrf_k=10`, 검색기별 `depth=20`, 최종 `top_k=5`
- Reranker 없음
- Parent-Child 확장 없음
- 컨텍스트 자르기 없음
- 질의 분류·질문 재작성·모호성 처리 없음

A안에서 하지 않는 것:

- Evidence Pack 생성
- Answer Skeleton 생성
- Fact Sheet 생성
- 사전 Fact Index 조회
- 사실 추출·병합·충돌 판정

`format_raw_top5_for_llm()`은 검색 결과의 원본 청크 딕셔너리를 순위와 함께 JSON으로 직렬화할 뿐입니다. 사실을 재구성하거나 요약하는 중간 단계가 아닙니다.


## 0. Colab 실행 순서

1. 런타임 유형은 기본 CPU로도 실행할 수 있습니다.
2. 아래 의존성 셀과 Elasticsearch 준비 셀을 위에서부터 실행합니다.
3. `KDIC_output(1).zip`을 업로드합니다.
4. **두 번째 실행부터** 캐시 업로드 셀에서 `kdic_dense_structured_v2_embeddings.jsonl`을 업로드합니다.
5. Colab 보안 비밀에 `HCX_API_KEY`를 등록하거나 입력창에 키를 입력합니다.
6. Dense 준비 셀은 업로드한 캐시를 검증하고 `valid=전체, missing=0`일 때 그대로 불러옵니다.
7. 최초 한 번만 `CREATE_DENSE_CACHE_ONCE=True`로 바꿔 문서 임베딩을 생성하며, 완성 파일은 브라우저로 자동 다운로드됩니다.
8. 마지막 질문 입력창에서 질문을 전송합니다.

Google Drive는 사용하지 않습니다. 캐시 파일은 팀원도 같은 파일을 내려받아 각자의 Colab `/content`로 업로드해 공통 사용합니다.

Elasticsearch는 이전 노트북과 충돌하지 않도록 전용 포트 `9220`을 사용합니다. 셀이 실패하면 마지막 로그가 자동으로 출력됩니다.


## 1. 의존성 설치

Elasticsearch 서버는 `8.15.3`, Python 클라이언트는 실제 배포되어 있는 같은 minor 계열의 `8.15.1`을 사용합니다. `elasticsearch==8.15.3`이라는 Python 패키지는 배포되어 있지 않으므로 해당 핀을 사용하면 설치 셀에서 바로 실패합니다.


In [ ]:
!pip -q install "openai>=1.68,<2" "elasticsearch==8.15.1" "tqdm>=4.66,<5" "ipywidgets>=8.1,<9"


## 2. Elasticsearch 8.15.3 + Nori 준비

이 셀은 Colab에서 자주 발생하는 다음 문제를 피하도록 구성했습니다.

- Elasticsearch를 root로 실행해서 발생하는 시작 실패
- 일반 사용자가 `/content`에 PID 파일을 쓰지 못하는 권한 오류
- 노트북 재실행 때 `elasticsearch.yml` 설정이 계속 중복되는 문제
- 기존 9200 포트 프로세스와의 충돌
- 부분 다운로드·부분 압축 해제로 인한 실행 파일 손상
- Nori 플러그인이 없는 상태에서 인덱스를 생성하는 문제
- Colab cgroup v2 경로가 샌드박스 밖으로 해석되어 `AccessControlException`이 발생하는 문제

설치 폴더, 데이터, 로그, PID 파일을 모두 A안 전용 경로로 분리합니다. Colab의 cgroup 경로는
Elasticsearch 컨테이너 실행 방식과 동일하게 루트(`/`)로 명시하여, Elasticsearch가
`/sys/fs/cgroup/../../jupyter-children/cpu.stat` 같은 잘못된 경로를 읽지 않도록 합니다.


In [ ]:
%%bash
set -Eeuo pipefail

ES_VERSION="8.15.3"
ES_USER="kdic_es_a"
INSTALL_ROOT="/content/kdic_es_a_dist"
ES_HOME="${INSTALL_ROOT}/elasticsearch-${ES_VERSION}"
ES_RUNTIME="/content/kdic_es_a_runtime"
ES_ARCHIVE="${INSTALL_ROOT}/elasticsearch-${ES_VERSION}.tar.gz"
ES_PID_FILE="${ES_RUNTIME}/elasticsearch.pid"
ES_LOG_FILE="${ES_RUNTIME}/logs/kdic-a.log"
ES_HTTP_URL="http://127.0.0.1:9220"

show_diagnostics() {
  echo "[Elasticsearch 진단]" >&2
  if [ -f "${ES_PID_FILE}" ]; then
    echo "PID file: $(cat "${ES_PID_FILE}" 2>/dev/null || true)" >&2
  fi
  if [ -f "${ES_LOG_FILE}" ]; then
    tail -n 160 "${ES_LOG_FILE}" >&2 || true
  elif [ -d "${ES_RUNTIME}/logs" ]; then
    tail -n 160 "${ES_RUNTIME}"/logs/*.log >&2 2>/dev/null || true
  fi
}
trap show_diagnostics ERR

case "$(uname -m)" in
  x86_64) ES_ARCH="x86_64" ;;
  aarch64|arm64) ES_ARCH="aarch64" ;;
  *) echo "지원하지 않는 CPU 아키텍처: $(uname -m)" >&2; exit 1 ;;
esac

mkdir -p "${INSTALL_ROOT}" "${ES_RUNTIME}/data" "${ES_RUNTIME}/logs" "${ES_RUNTIME}/tmp"

if ! id "${ES_USER}" >/dev/null 2>&1; then
  useradd --system --create-home --home-dir "/content/${ES_USER}" --shell /usr/sbin/nologin "${ES_USER}"
fi

if curl -fsS "${ES_HTTP_URL}" >/dev/null 2>&1; then
  RUNNING_VERSION="$(curl -fsS "${ES_HTTP_URL}" | python3 -c 'import json,sys; print(json.load(sys.stdin)["version"]["number"])')"
  if [ "${RUNNING_VERSION}" != "${ES_VERSION}" ]; then
    echo "9220 포트에 Elasticsearch ${RUNNING_VERSION}가 실행 중입니다. Colab 런타임을 재시작하세요." >&2
    exit 1
  fi
  if ! curl -fsS "${ES_HTTP_URL}/_nodes/plugins" | python3 -c 'import json,sys; d=json.load(sys.stdin); assert any(p.get("name")=="analysis-nori" for n in d["nodes"].values() for p in n.get("plugins", []))'; then
    echo "실행 중인 9220 Elasticsearch에 analysis-nori가 없습니다. Colab 런타임을 재시작하세요." >&2
    exit 1
  fi
  echo "Elasticsearch ${RUNNING_VERSION} + analysis-nori 재사용"
  exit 0
fi

# A안 전용 PID는 남아 있지만 HTTP가 열리지 않으면 해당 프로세스만 정리한 뒤 재시작합니다.
if [ -f "${ES_PID_FILE}" ]; then
  OLD_PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
  if [ -n "${OLD_PID}" ] && kill -0 "${OLD_PID}" 2>/dev/null; then
    kill "${OLD_PID}" 2>/dev/null || true
    for _ in $(seq 1 15); do
      if ! kill -0 "${OLD_PID}" 2>/dev/null; then
        break
      fi
      sleep 1
    done
    if kill -0 "${OLD_PID}" 2>/dev/null; then
      kill -9 "${OLD_PID}" 2>/dev/null || true
    fi
  fi
  rm -f "${ES_PID_FILE}"
fi

if [ ! -x "${ES_HOME}/bin/elasticsearch" ]; then
  DOWNLOAD_URL="https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-${ES_ARCH}.tar.gz"
  TEMP_ARCHIVE="${ES_ARCHIVE}.part"
  rm -f "${TEMP_ARCHIVE}"
  curl -fL --retry 5 --retry-delay 3 --connect-timeout 20 \
    "${DOWNLOAD_URL}" -o "${TEMP_ARCHIVE}"
  tar -tzf "${TEMP_ARCHIVE}" >/dev/null
  mv "${TEMP_ARCHIVE}" "${ES_ARCHIVE}"
  tar -xzf "${ES_ARCHIVE}" -C "${INSTALL_ROOT}"
fi

chown -R "${ES_USER}:${ES_USER}" "${ES_HOME}" "${ES_RUNTIME}" "/content/${ES_USER}"

if ! runuser -u "${ES_USER}" -- "${ES_HOME}/bin/elasticsearch-plugin" list | grep -qx "analysis-nori"; then
  runuser -u "${ES_USER}" -- "${ES_HOME}/bin/elasticsearch-plugin" install --batch analysis-nori
fi

CONFIG_FILE="${ES_HOME}/config/elasticsearch.yml"
python3 - "${CONFIG_FILE}" "${ES_RUNTIME}" <<'PY'
from pathlib import Path
import sys

config_path = Path(sys.argv[1])
runtime = Path(sys.argv[2])
config_path.write_text(
    "\n".join([
        "cluster.name: kdic-colab-answer-a",
        "node.name: kdic-colab-answer-a-node",
        f"path.data: {runtime / 'data'}",
        f"path.logs: {runtime / 'logs'}",
        "network.host: 127.0.0.1",
        "http.port: 9220",
        "transport.port: 9320",
        "discovery.type: single-node",
        "xpack.security.enabled: false",
        "xpack.security.enrollment.enabled: false",
        "xpack.security.http.ssl.enabled: false",
        "xpack.security.transport.ssl.enabled: false",
        "xpack.ml.enabled: false",
        "ingest.geoip.downloader.enabled: false",
        "cluster.routing.allocation.disk.threshold_enabled: false",
        "node.store.allow_mmap: false",
        "bootstrap.memory_lock: false",
        "",
    ]),
    encoding="utf-8",
)
PY
chown "${ES_USER}:${ES_USER}" "${CONFIG_FILE}"

if [ -f "${ES_PID_FILE}" ]; then
  OLD_PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
  if [ -n "${OLD_PID}" ] && kill -0 "${OLD_PID}" 2>/dev/null; then
    echo "기존 Elasticsearch PID ${OLD_PID}의 시작을 기다립니다."
  else
    rm -f "${ES_PID_FILE}"
  fi
fi

if [ ! -f "${ES_PID_FILE}" ]; then
  runuser -u "${ES_USER}" -- env \
    ES_JAVA_OPTS="-Xms512m -Xmx512m -Djava.io.tmpdir=${ES_RUNTIME}/tmp -Des.cgroups.hierarchy.override=/" \
    "${ES_HOME}/bin/elasticsearch" -d -p "${ES_PID_FILE}"
fi

for _ in $(seq 1 120); do
  if curl -fsS "${ES_HTTP_URL}" >/dev/null 2>&1; then
    break
  fi
  if [ -f "${ES_PID_FILE}" ]; then
    PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
    if [ -n "${PID}" ] && ! kill -0 "${PID}" 2>/dev/null; then
      echo "Elasticsearch 프로세스가 시작 중 종료되었습니다." >&2
      exit 1
    fi
  fi
  sleep 1
done

curl -fsS "${ES_HTTP_URL}" >/dev/null
curl -fsS "${ES_HTTP_URL}/_nodes/jvm" | python3 -c '
import json, sys
data = json.load(sys.stdin)
args = [arg for node in data["nodes"].values() for arg in node["jvm"].get("input_arguments", [])]
assert "-Des.cgroups.hierarchy.override=/" in args, args
'
curl -fsS "${ES_HTTP_URL}/_nodes/plugins" | python3 -c 'import json,sys; d=json.load(sys.stdin); assert any(p.get("name")=="analysis-nori" for n in d["nodes"].values() for p in n.get("plugins", []))'
curl -fsS -X POST "${ES_HTTP_URL}/_analyze" \
  -H 'Content-Type: application/json' \
  -d '{"tokenizer":{"type":"nori_tokenizer","decompound_mode":"none"},"text":"예금자보호제도"}' >/dev/null

echo "Elasticsearch ${ES_VERSION} + analysis-nori 준비 완료: ${ES_HTTP_URL}"


## 3. 공통 설정과 라이브러리


In [ ]:
from __future__ import annotations

import getpass
import hashlib
import json
import math
import os
import re
import shutil
import zipfile
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Literal

import ipywidgets as widgets
import numpy as np
from elasticsearch import Elasticsearch, helpers
from IPython.display import JSON, Markdown, clear_output, display
from openai import BadRequestError, OpenAI
from tqdm.auto import tqdm

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass


# ---------- 데이터 / 캐시 ----------
# 경로를 직접 지정하지 않으면 ZIP 업로드 창이 열립니다.
DATA_SOURCE: str | None = None
DENSE_CACHE_FILENAME = "kdic_dense_structured_v2_embeddings.jsonl"
DENSE_CACHE_PATH = Path("/content") / DENSE_CACHE_FILENAME

# ---------- HCX ----------
HCX_BASE_URL = "https://clovastudio.stream.ntruss.com/v1/openai"
HCX_EMBEDDING_MODEL = "bge-m3"
HCX_CHAT_MODEL = "HCX-005"
HCX_ENCODING_FORMAT = "float"
HCX_REQUEST_TIMEOUT = 120.0
HCX_MAX_RETRIES = 4

# ---------- 확정 검색 조건 ----------
DENSE_WEIGHT = 0.7
BM25_WEIGHT = 0.3
RRF_K = 10
CANDIDATE_DEPTH = 20
FINAL_TOP_K = 5

# ---------- 버전 / Elasticsearch ----------
DENSE_INPUT_VERSION = "kdic-dense-structured-v2-title-section-content-newline"
DENSE_CACHE_VERSION = "kdic-hcx-dense-structured-v2-cache-v1"
ES_EXPECTED_VERSION = "8.15.3"
ES_URL = "http://127.0.0.1:9220"
ES_ANALYZER_NAME = "kdic_nori_none"
ES_INDEX_SCHEMA_VERSION = "kdic-bm25-nori-none-a-v2"
FORCE_REBUILD_BM25_INDEX = False

assert math.isclose(DENSE_WEIGHT + BM25_WEIGHT, 1.0)
assert RRF_K > 0 and CANDIDATE_DEPTH > 0 and FINAL_TOP_K > 0

print({
    "answer_method": "A_RAW_TOP5",
    "dense": "HCX bge-m3 Dense-structured-v2",
    "sparse": "Elasticsearch BM25 + Nori-none",
    "weights": [DENSE_WEIGHT, BM25_WEIGHT],
    "rrf_k": RRF_K,
    "candidate_depth": CANDIDATE_DEPTH,
    "final_top_k": FINAL_TOP_K,
    "reranker": False,
    "parent_child": False,
    "evidence_pack": False,
    "fact_index": False,
    "fact_sheet": False,
})


## 4. KDIC ZIP 업로드와 청크 로딩

필수 파일은 ZIP 내부의 `processed/chunks.jsonl`입니다. 기존 `chunk_embeddings_hcx.jsonl`은 `content` 중심 임베딩이므로 Dense-structured-v2 캐시로 사용하지 않습니다.


In [ ]:
def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"JSONL 파싱 실패: {path}, line={line_number}") from error
            if not isinstance(record, dict):
                raise TypeError(f"JSONL 레코드가 객체가 아닙니다: {path}, line={line_number}")
            records.append(record)
    return records


def _safe_extract_zip(zip_path: Path, destination: Path) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"손상된 ZIP 항목: {bad_member}")
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"안전하지 않은 ZIP 경로: {member.filename}")
        archive.extractall(destination)
    return destination


def _find_unique_file(root: Path, filename: str) -> Path:
    matches = list(root.rglob(filename))
    processed = [path for path in matches if path.parent.name == "processed"]
    candidates = processed or matches
    if not candidates:
        raise FileNotFoundError(f"{filename}을 찾지 못했습니다: {root}")
    if len(candidates) != 1:
        raise RuntimeError(f"{filename} 후보가 여러 개입니다: {candidates}")
    return candidates[0]


def resolve_data_source(configured_path: str | None) -> Path:
    if configured_path:
        path = Path(configured_path)
        if path.exists():
            return path
        raise FileNotFoundError(f"DATA_SOURCE 경로가 없습니다: {path}")

    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError(
            "DATA_SOURCE에 KDIC_output ZIP 또는 압축 해제 폴더 경로를 지정하세요."
        ) from error

    print("KDIC_output ZIP 파일을 업로드하세요.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError(f"ZIP 파일을 정확히 1개 업로드해야 합니다: {list(uploaded)}")
    return Path("/content") / zip_names[0]


def prepare_data_root(source: Path) -> Path:
    if source.is_dir():
        return source
    if not zipfile.is_zipfile(source):
        raise ValueError(f"ZIP 파일이 아닙니다: {source}")
    digest = hashlib.sha256(source.read_bytes()).hexdigest()[:16]
    destination = Path("/content/kdic_data_a") / digest
    marker = destination / ".ready"
    if marker.exists():
        return destination
    if destination.exists():
        shutil.rmtree(destination)
    _safe_extract_zip(source, destination)
    marker.write_text("ready", encoding="utf-8")
    return destination


def _clean_text(value: Any) -> str:
    text = str(value or "").replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_chunks(data_root: Path) -> list[dict[str, Any]]:
    chunks_path = _find_unique_file(data_root, "chunks.jsonl")
    chunks = _read_jsonl(chunks_path)
    if not chunks:
        raise RuntimeError("chunks.jsonl이 비어 있습니다.")

    chunk_ids = [str(chunk.get("chunk_id") or "").strip() for chunk in chunks]
    if any(not chunk_id for chunk_id in chunk_ids):
        raise RuntimeError("빈 chunk_id가 있습니다.")
    if len(chunk_ids) != len(set(chunk_ids)):
        raise RuntimeError("중복 chunk_id가 있습니다.")
    if any(not _clean_text(chunk.get("content")) for chunk in chunks):
        raise RuntimeError("본문이 비어 있는 청크가 있습니다.")
    return chunks


def build_dense_structured_v2_text(chunk: dict[str, Any]) -> str:
    parts = [
        _clean_text(chunk.get("title")),
        _clean_text(chunk.get("section_title")),
        _clean_text(chunk.get("content")),
    ]
    text = "\n".join(part for part in parts if part)
    if not text:
        raise ValueError(f"Dense 입력이 비었습니다: {chunk.get('chunk_id')}")
    return text


DATA_PATH = resolve_data_source(DATA_SOURCE)
DATA_ROOT = prepare_data_root(DATA_PATH)
CHUNKS = load_chunks(DATA_ROOT)
CHUNKS_BY_ID = {str(chunk["chunk_id"]): chunk for chunk in CHUNKS}

dataset_hash = hashlib.sha256()
for chunk in CHUNKS:
    dataset_hash.update(str(chunk["chunk_id"]).encode("utf-8"))
    dataset_hash.update(b"\0")
    dataset_hash.update(build_dense_structured_v2_text(chunk).encode("utf-8"))
    dataset_hash.update(b"\0")
DATASET_FINGERPRINT = dataset_hash.hexdigest()
ES_INDEX_NAME = f"kdic-bm25-nori-none-a-v2-{DATASET_FINGERPRINT[:12]}"

print("데이터 경로:", DATA_ROOT)
print("청크 수:", len(CHUNKS))
print("데이터 지문:", DATASET_FINGERPRINT[:16])
print("업무:", sorted({_clean_text(chunk.get("business_function")) for chunk in CHUNKS}))


## 4-1. 기존 Dense-structured-v2 캐시 업로드

**최초 생성 시에는 이 셀을 건너뜁니다.** 이미 만들어 둔 캐시를 재사용할 때만 실행하고,
정확히 `kdic_dense_structured_v2_embeddings.jsonl` 파일 하나를 업로드하세요.

업로드한 파일은 뒤의 Dense 준비 셀에서 청크 ID, 모델, 입력 구조 버전, 입력 SHA-256,
임베딩 차원을 검증합니다. 검증을 통과하지 못한 항목이 있더라도 기본 설정에서는 자동으로
재임베딩하지 않고 중단합니다.


In [ ]:
def upload_dense_cache_from_browser(
    target_path: Path = DENSE_CACHE_PATH,
    *,
    reuse_existing: bool = True,
) -> Path:
    if reuse_existing and target_path.is_file() and target_path.stat().st_size > 0:
        print("이미 런타임에 있는 캐시를 재사용합니다:", target_path)
        return target_path

    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError(
            f"Colab이 아니면 {target_path} 경로에 {DENSE_CACHE_FILENAME}을 직접 복사하세요."
        ) from error

    print(f"{DENSE_CACHE_FILENAME} 파일 하나를 업로드하세요.")
    uploaded = files.upload()
    if set(uploaded) != {DENSE_CACHE_FILENAME}:
        raise RuntimeError(
            "업로드 파일명이 정확하지 않습니다. "
            f"expected={DENSE_CACHE_FILENAME}, uploaded={list(uploaded)}"
        )

    payload = uploaded[DENSE_CACHE_FILENAME]
    if not payload:
        raise RuntimeError("업로드한 Dense 캐시 파일이 비어 있습니다.")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_bytes(payload)
    print(f"Dense 캐시 업로드 완료: {target_path} ({target_path.stat().st_size:,} bytes)")
    return target_path


# 두 번째 런타임부터 이 셀을 실행합니다. 최초 생성 시에는 셀 자체를 건너뛰세요.
UPLOADED_DENSE_CACHE_PATH = upload_dense_cache_from_browser()


## 5. Elasticsearch 연결 검증과 BM25 + Nori-none 인덱스

인덱스 이름에는 데이터 지문이 포함됩니다. 같은 데이터를 다시 실행하면 기존 정상 인덱스를 재사용하고, 문서 수·스키마·데이터 지문이 다르면 해당 A안 인덱스만 다시 만듭니다.


In [ ]:
def connect_elasticsearch() -> Elasticsearch:
    client = Elasticsearch(
        ES_URL,
        request_timeout=120,
        max_retries=5,
        retry_on_timeout=True,
    )
    try:
        info = client.info()
    except Exception as error:
        raise RuntimeError(
            "Elasticsearch 연결 실패입니다. 2번 준비 셀의 마지막 로그를 확인하세요. "
            f"원인={type(error).__name__}: {error}"
        ) from error

    running_version = str(info["version"]["number"])
    if running_version != ES_EXPECTED_VERSION:
        raise RuntimeError(
            f"Elasticsearch 버전 불일치: running={running_version}, expected={ES_EXPECTED_VERSION}"
        )

    nodes = client.nodes.info(metric="plugins")
    plugin_names = {
        str(plugin.get("name") or "")
        for node in nodes["nodes"].values()
        for plugin in node.get("plugins", [])
    }
    if "analysis-nori" not in plugin_names:
        raise RuntimeError(f"analysis-nori 플러그인이 없습니다: {sorted(plugin_names)}")
    return client


def _index_is_reusable(client: Elasticsearch) -> bool:
    if FORCE_REBUILD_BM25_INDEX:
        return False
    if not client.indices.exists(index=ES_INDEX_NAME):
        return False
    count = int(client.count(index=ES_INDEX_NAME)["count"])
    mapping = client.indices.get_mapping(index=ES_INDEX_NAME)
    metadata = mapping[ES_INDEX_NAME]["mappings"].get("_meta", {})
    return (
        count == len(CHUNKS)
        and metadata.get("schema_version") == ES_INDEX_SCHEMA_VERSION
        and metadata.get("dataset_fingerprint") == DATASET_FINGERPRINT
    )


def prepare_bm25_nori_none_index(client: Elasticsearch) -> None:
    if _index_is_reusable(client):
        print(f"기존 BM25 인덱스 재사용: {ES_INDEX_NAME}")
        return

    if client.indices.exists(index=ES_INDEX_NAME):
        client.indices.delete(index=ES_INDEX_NAME)

    client.indices.create(
        index=ES_INDEX_NAME,
        settings={
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "similarity": {
                "kdic_bm25": {
                    "type": "BM25",
                    "k1": 1.2,
                    "b": 0.75,
                }
            },
            "analysis": {
                "tokenizer": {
                    "kdic_nori_none_tokenizer": {
                        "type": "nori_tokenizer",
                        "decompound_mode": "none",
                    }
                },
                "analyzer": {
                    ES_ANALYZER_NAME: {
                        "type": "custom",
                        "tokenizer": "kdic_nori_none_tokenizer",
                    }
                },
            },
        },
        mappings={
            "_meta": {
                "schema_version": ES_INDEX_SCHEMA_VERSION,
                "dataset_fingerprint": DATASET_FINGERPRINT,
            },
            "properties": {
                "chunk_id": {"type": "keyword"},
                "search_text": {
                    "type": "text",
                    "analyzer": ES_ANALYZER_NAME,
                    "search_analyzer": ES_ANALYZER_NAME,
                    "similarity": "kdic_bm25",
                },
            },
        },
    )

    actions = (
        {
            "_op_type": "index",
            "_index": ES_INDEX_NAME,
            "_id": str(chunk["chunk_id"]),
            "_source": {
                "chunk_id": str(chunk["chunk_id"]),
                "search_text": build_dense_structured_v2_text(chunk),
            },
        }
        for chunk in CHUNKS
    )
    bulk_client = client.options(request_timeout=120)
    success, errors = helpers.bulk(
        bulk_client,
        actions,
        chunk_size=100,
        max_retries=4,
        initial_backoff=1,
        max_backoff=8,
        raise_on_error=False,
        raise_on_exception=False,
    )
    client.indices.refresh(index=ES_INDEX_NAME)

    if errors:
        preview = json.dumps(errors[:3], ensure_ascii=False, default=str)[:3000]
        raise RuntimeError(f"BM25 인덱싱 실패 {len(errors)}건: {preview}")
    if int(success) != len(CHUNKS):
        raise RuntimeError(f"BM25 인덱싱 건수 불일치: success={success}, chunks={len(CHUNKS)}")

    actual_count = int(client.count(index=ES_INDEX_NAME)["count"])
    if actual_count != len(CHUNKS):
        raise RuntimeError(f"BM25 저장 건수 불일치: index={actual_count}, chunks={len(CHUNKS)}")

    analysis = client.indices.analyze(
        index=ES_INDEX_NAME,
        analyzer=ES_ANALYZER_NAME,
        text="예금자보호제도",
    )
    if not analysis.get("tokens"):
        raise RuntimeError("Nori 분석 결과가 비어 있습니다.")


ES = connect_elasticsearch()
prepare_bm25_nori_none_index(ES)

print("Elasticsearch:", ES.info()["version"]["number"])
print("BM25 인덱스:", ES_INDEX_NAME)
print("BM25 문서 수:", ES.count(index=ES_INDEX_NAME)["count"])
print("Nori-none 토큰:", [
    token["token"]
    for token in ES.indices.analyze(
        index=ES_INDEX_NAME,
        analyzer=ES_ANALYZER_NAME,
        text="예금자보호제도",
    )["tokens"]
])


## 6. HCX 클라이언트와 Dense-structured-v2 임베딩 캐시


In [ ]:
def load_hcx_api_key() -> str:
    key: str | None = None
    try:
        from google.colab import userdata
        key = userdata.get("HCX_API_KEY")
    except Exception:
        key = os.environ.get("HCX_API_KEY")
    if not key:
        key = getpass.getpass("HCX_API_KEY: ")

    key = str(key or "").strip()
    if not key:
        raise ValueError("HCX_API_KEY가 비어 있습니다.")
    if key.lower().startswith("bearer "):
        raise ValueError("HCX_API_KEY 앞에 'Bearer '를 붙이지 마세요.")
    if any(character.isspace() for character in key):
        raise ValueError("HCX_API_KEY 안에 공백 또는 줄바꿈이 있습니다.")
    return key


HCX_API_KEY = load_hcx_api_key()
HCX_CLIENT = OpenAI(
    api_key=HCX_API_KEY,
    base_url=HCX_BASE_URL,
    timeout=HCX_REQUEST_TIMEOUT,
    max_retries=HCX_MAX_RETRIES,
)


def embed_hcx_single(text: str) -> np.ndarray:
    cleaned = _clean_text(text)
    if not cleaned:
        raise ValueError("임베딩 입력이 비어 있습니다.")
    response = HCX_CLIENT.embeddings.create(
        model=HCX_EMBEDDING_MODEL,
        input=cleaned,
        encoding_format=HCX_ENCODING_FORMAT,
    )
    if len(response.data) != 1:
        raise RuntimeError(f"단일 임베딩 응답 개수가 1이 아닙니다: {len(response.data)}")
    vector = np.asarray(response.data[0].embedding, dtype=np.float32)
    if vector.ndim != 1 or vector.size == 0:
        raise RuntimeError(f"잘못된 임베딩 shape: {vector.shape}")
    if not np.all(np.isfinite(vector)):
        raise RuntimeError("임베딩에 NaN 또는 무한대가 있습니다.")
    return vector


print("HCX 클라이언트 준비 완료")
print("Dense 입력 예시:\n", build_dense_structured_v2_text(CHUNKS[0])[:500])


In [ ]:
def structured_input_sha256(chunk: dict[str, Any]) -> str:
    text = build_dense_structured_v2_text(chunk)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _load_valid_dense_cache(path: Path) -> dict[str, dict[str, Any]]:
    if not path.exists():
        return {}
    valid: dict[str, dict[str, Any]] = {}
    for record in _read_jsonl(path):
        chunk_id = str(record.get("chunk_id") or "")
        chunk = CHUNKS_BY_ID.get(chunk_id)
        if chunk is None:
            continue
        if record.get("model") != HCX_EMBEDDING_MODEL:
            continue
        if record.get("input_version") != DENSE_INPUT_VERSION:
            continue
        if record.get("cache_version") != DENSE_CACHE_VERSION:
            continue
        if record.get("input_sha256") != structured_input_sha256(chunk):
            continue
        vector = np.asarray(record.get("embedding"), dtype=np.float32)
        if vector.ndim != 1 or vector.size == 0 or not np.all(np.isfinite(vector)):
            continue
        if int(record.get("dimensions") or 0) != vector.size:
            continue
        valid[chunk_id] = record
    return valid


def _write_dense_cache_atomic(path: Path, records_by_id: dict[str, dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as file:
        for chunk in CHUNKS:
            record = records_by_id.get(str(chunk["chunk_id"]))
            if record is not None:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")
    os.replace(temp_path, path)


def prepare_dense_embeddings(
    cache_path: Path = DENSE_CACHE_PATH,
    checkpoint_every: int = 5,
    allow_generate_missing: bool = False,
) -> tuple[np.ndarray, list[str]]:
    cache = _load_valid_dense_cache(cache_path)
    missing = [chunk for chunk in CHUNKS if str(chunk["chunk_id"]) not in cache]
    print(f"Dense cache: valid={len(cache)}, missing={len(missing)}")

    if missing and not allow_generate_missing:
        raise RuntimeError(
            "Dense 캐시에 유효한 문서 임베딩이 부족하므로 자동 생성을 중단했습니다. "
            f"valid={len(cache)}, missing={len(missing)}. "
            "기존 캐시 파일을 올바르게 업로드하거나, 최초 생성일 때만 "
            "CREATE_DENSE_CACHE_ONCE=True로 바꾼 뒤 이 셀을 다시 실행하세요."
        )

    try:
        for index, chunk in enumerate(
            tqdm(missing, desc="Dense-structured-v2 embedding"),
            start=1,
        ):
            chunk_id = str(chunk["chunk_id"])
            input_text = build_dense_structured_v2_text(chunk)
            vector = embed_hcx_single(input_text)
            cache[chunk_id] = {
                "chunk_id": chunk_id,
                "model": HCX_EMBEDDING_MODEL,
                "encoding_format": HCX_ENCODING_FORMAT,
                "input_version": DENSE_INPUT_VERSION,
                "input_sha256": hashlib.sha256(input_text.encode("utf-8")).hexdigest(),
                "cache_version": DENSE_CACHE_VERSION,
                "dimensions": int(vector.size),
                "embedding": vector.tolist(),
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            if index % checkpoint_every == 0:
                _write_dense_cache_atomic(cache_path, cache)
    finally:
        if cache:
            _write_dense_cache_atomic(cache_path, cache)

    ordered_vectors: list[np.ndarray] = []
    dimensions: set[int] = set()
    chunk_ids: list[str] = []
    for chunk in CHUNKS:
        chunk_id = str(chunk["chunk_id"])
        record = cache.get(chunk_id)
        if record is None:
            raise RuntimeError(f"Dense 캐시 누락: {chunk_id}")
        vector = np.asarray(record["embedding"], dtype=np.float32)
        norm = float(np.linalg.norm(vector))
        if norm == 0.0:
            raise RuntimeError(f"영벡터 임베딩: {chunk_id}")
        ordered_vectors.append(vector / norm)
        dimensions.add(int(vector.size))
        chunk_ids.append(chunk_id)

    if len(dimensions) != 1:
        raise RuntimeError(f"임베딩 차원 불일치: {dimensions}")
    return np.vstack(ordered_vectors), chunk_ids


def download_dense_cache_to_browser(
    cache_path: Path = DENSE_CACHE_PATH,
) -> None:
    if not cache_path.is_file() or cache_path.stat().st_size == 0:
        raise FileNotFoundError(f"다운로드할 Dense 캐시가 없습니다: {cache_path}")
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError(f"Colab 외부에서는 이 파일을 직접 가져가세요: {cache_path}") from error
    print(f"Dense 캐시 다운로드를 시작합니다: {cache_path.name}")
    files.download(str(cache_path))


# 최초 캐시를 만드는 단 한 번만 True로 바꾸세요.
# 이후 A~E 노트북에서는 False를 유지하고 캐시 업로드 셀을 실행합니다.
CREATE_DENSE_CACHE_ONCE = False

DENSE_MATRIX, DENSE_CHUNK_IDS = prepare_dense_embeddings(
    allow_generate_missing=CREATE_DENSE_CACHE_ONCE,
)
DENSE_DIMENSION = int(DENSE_MATRIX.shape[1])

print("Dense matrix:", DENSE_MATRIX.shape)
print("Dense cache:", DENSE_CACHE_PATH)
if CREATE_DENSE_CACHE_ONCE:
    download_dense_cache_to_browser()
    print("다운로드한 파일을 보관하고 A~E 실험에서 공통으로 업로드해 사용하세요.")
else:
    print("문서 임베딩 API 호출 없이 업로드된 Dense 캐시를 사용했습니다.")


## 7. Dense + BM25 + Weighted RRF 검색

결합 점수는 원점수 합이 아니라 순위 기반입니다.

$$
\operatorname{WRRF}(d)=
\frac{0.7}{10+r_{dense}(d)}+
\frac{0.3}{10+r_{bm25}(d)}
$$

`depth=20`은 각 검색기가 결합 전에 반환하는 후보 수입니다. 최종 답변에는 결합 결과 상위 5개만 사용합니다.


In [ ]:
def _normalize_vector(vector: np.ndarray) -> np.ndarray:
    vector = np.asarray(vector, dtype=np.float32)
    norm = float(np.linalg.norm(vector))
    if norm == 0.0:
        raise RuntimeError("질문 임베딩이 영벡터입니다.")
    return vector / norm


def dense_search(question: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    query_vector = _normalize_vector(embed_hcx_single(question))
    if query_vector.shape != (DENSE_DIMENSION,):
        raise RuntimeError(
            f"질문 임베딩 차원 불일치: query={query_vector.shape}, stored={DENSE_DIMENSION}"
        )
    scores = DENSE_MATRIX @ query_vector
    limit = min(depth, len(DENSE_CHUNK_IDS))
    candidate_indices = np.argpartition(-scores, limit - 1)[:limit]
    ordered_indices = sorted(
        candidate_indices.tolist(),
        key=lambda index: (-float(scores[index]), DENSE_CHUNK_IDS[index]),
    )
    return [
        {
            "chunk_id": DENSE_CHUNK_IDS[index],
            "score": float(scores[index]),
            "rank": rank,
        }
        for rank, index in enumerate(ordered_indices, start=1)
    ]


def bm25_search(question: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    response = ES.search(
        index=ES_INDEX_NAME,
        size=depth,
        query={"match": {"search_text": {"query": question}}},
    )
    results: list[dict[str, Any]] = []
    for rank, hit in enumerate(response["hits"]["hits"], start=1):
        chunk_id = str(hit["_source"]["chunk_id"])
        if chunk_id not in CHUNKS_BY_ID:
            raise RuntimeError(f"BM25 결과의 chunk_id가 원본에 없습니다: {chunk_id}")
        results.append({
            "chunk_id": chunk_id,
            "score": float(hit["_score"]),
            "rank": rank,
        })
    return results


def weighted_rrf(
    dense_results: list[dict[str, Any]],
    bm25_results: list[dict[str, Any]],
    *,
    dense_weight: float = DENSE_WEIGHT,
    bm25_weight: float = BM25_WEIGHT,
    rrf_k: int = RRF_K,
    top_k: int = FINAL_TOP_K,
) -> list[dict[str, Any]]:
    if not math.isclose(dense_weight + bm25_weight, 1.0):
        raise ValueError("Dense/BM25 가중치 합은 1이어야 합니다.")
    if rrf_k <= 0 or top_k <= 0:
        raise ValueError("rrf_k와 top_k는 1 이상이어야 합니다.")

    candidates: dict[str, dict[str, Any]] = {}

    for result in dense_results:
        chunk_id = str(result["chunk_id"])
        row = candidates.setdefault(chunk_id, {
            "chunk_id": chunk_id,
            "rrf_score": 0.0,
            "dense_rank": None,
            "dense_score": None,
            "bm25_rank": None,
            "bm25_score": None,
        })
        rank = int(result["rank"])
        row["dense_rank"] = rank
        row["dense_score"] = float(result["score"])
        row["rrf_score"] += dense_weight / (rrf_k + rank)

    for result in bm25_results:
        chunk_id = str(result["chunk_id"])
        row = candidates.setdefault(chunk_id, {
            "chunk_id": chunk_id,
            "rrf_score": 0.0,
            "dense_rank": None,
            "dense_score": None,
            "bm25_rank": None,
            "bm25_score": None,
        })
        rank = int(result["rank"])
        row["bm25_rank"] = rank
        row["bm25_score"] = float(result["score"])
        row["rrf_score"] += bm25_weight / (rrf_k + rank)

    infinity = float("inf")
    ordered = sorted(
        candidates.values(),
        key=lambda row: (
            -float(row["rrf_score"]),
            min(row["dense_rank"] or infinity, row["bm25_rank"] or infinity),
            row["dense_rank"] or infinity,
            row["bm25_rank"] or infinity,
            row["chunk_id"],
        ),
    )

    final_results: list[dict[str, Any]] = []
    for final_rank, row in enumerate(ordered[:top_k], start=1):
        final_results.append({
            **row,
            "rank": final_rank,
            "chunk": CHUNKS_BY_ID[row["chunk_id"]],
        })
    return final_results


def hybrid_search(question: str) -> list[dict[str, Any]]:
    cleaned = _clean_text(question)
    if not cleaned:
        raise ValueError("질문이 비어 있습니다.")
    dense_results = dense_search(cleaned, CANDIDATE_DEPTH)
    bm25_results = bm25_search(cleaned, CANDIDATE_DEPTH)
    results = weighted_rrf(
        dense_results,
        bm25_results,
        dense_weight=DENSE_WEIGHT,
        bm25_weight=BM25_WEIGHT,
        rrf_k=RRF_K,
        top_k=FINAL_TOP_K,
    )
    if not results:
        raise RuntimeError("Hybrid 검색 결과가 없습니다.")
    return results


print("Hybrid retriever 준비 완료")


## 8. A안 — Top 5 원본 청크를 직접 HCX-005에 전달

`format_raw_top5_for_llm()`은 검색된 청크 원본 딕셔너리에 `retrieval_rank`만 붙여 JSON 문자열로 만듭니다.

- 새로운 사실 스키마를 만들지 않습니다.
- 내용을 추출·요약·병합하지 않습니다.
- 본문을 자르지 않습니다.
- Dense/BM25 점수는 답변 모델에 주지 않습니다.
- 쉬운 답변과 전문가 답변은 동일한 검색 결과 객체를 사용합니다.

URL은 모델이 생성하지 않고 Top 5 원본 청크에서 규칙 기반으로 답변 아래에 붙입니다.


In [ ]:
COMMON_ANSWER_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 질의응답 시스템입니다.

반드시 지킬 규칙:
1. 제공된 Top 5 원본 청크에 명시된 내용만 사용하여 한국어로 답하세요.
2. 질문에 먼저 직접 답한 뒤, 필요한 조건·예외·금액·기간·절차를 설명하세요.
3. 청크에 없는 사실을 추정하거나 일반상식으로 보완하지 마세요.
4. 핵심 주장 뒤에는 검색 순위에 대응하는 [C1]~[C5]를 표시하세요.
5. 원문끼리 내용이 다르면 한쪽을 임의로 선택하지 말고 차이를 설명하세요.
6. Top 5만으로 확인할 수 없는 내용은 확인할 수 없다고 명시하세요.
7. URL, 전화번호, 추천 질문, 추천 키워드를 답변 본문에 작성하지 마세요.
8. 검색 점수와 내부 구현을 답변 본문에 쓰지 마세요.
""".strip()

EASY_STYLE_PROMPT = """
쉬운 답변 작성 방식:
- 일반 사용자가 바로 이해할 수 있는 표현을 사용하세요.
- 첫 문단에서 결론을 짧게 말하세요.
- 전문용어가 필요하면 바로 뒤에서 쉬운 말로 풀이하세요.
- 조건, 예외, 금액, 기간, 신청 절차처럼 행동에 영향을 주는 사실은 쉬운 설명이라는 이유로 생략하지 마세요.
- 긴 문장보다 짧은 문장과 필요한 목록을 사용하세요.
""".strip()

DETAILED_STYLE_PROMPT = """
자세한 답변 작성 방식:
- 제도와 절차를 정확한 용어로 설명하세요.
- 적용 대상, 성립 조건, 제외·예외, 금액·기한, 제출·처리 절차를 원문 범위 안에서 구분하세요.
- 서로 다른 청크의 관계를 명확히 설명하되, 원문에 없는 법적 해석을 추가하지 마세요.
- 필요하면 소제목과 목록을 사용하여 구조적으로 작성하세요.
""".strip()


def format_raw_top5_for_llm(search_results: list[dict[str, Any]]) -> str:
    raw_chunks: list[dict[str, Any]] = []
    for result in search_results:
        original_chunk = result["chunk"]
        if not isinstance(original_chunk, dict):
            raise TypeError("원본 청크가 dict가 아닙니다.")
        raw_chunks.append({
            **original_chunk,
            "retrieval_rank": int(result["rank"]),
        })
    return json.dumps(raw_chunks, ensure_ascii=False, indent=2, default=str)


def _remove_model_generated_urls(answer: str) -> str:
    answer = re.sub(r"\[([^\]]+)\]\(https?://[^)]+\)", r"\1", answer)
    answer = re.sub(r"https?://[^\s)\]}>]+", "", answer)
    answer = re.sub(r"[ \t]+\n", "\n", answer)
    answer = re.sub(r"\n{3,}", "\n\n", answer)
    return answer.strip()


def _remove_invalid_chunk_citations(answer: str, result_count: int) -> str:
    allowed = {f"C{index}" for index in range(1, result_count + 1)}

    def replace(match: re.Match[str]) -> str:
        citation = f"C{match.group(1)}"
        return match.group(0) if citation in allowed else ""

    answer = re.sub(r"\[C(\d+)\]", replace, answer)
    answer = re.sub(r"[ \t]+\n", "\n", answer)
    answer = re.sub(r"\n{3,}", "\n\n", answer)
    return answer.strip()


def generate_raw_top5_answer(
    question: str,
    search_results: list[dict[str, Any]],
    *,
    answer_mode: Literal["easy", "detailed"] = "easy",
) -> str:
    if not search_results:
        return "검색된 공식 근거가 없어 현재 데이터로 답할 수 없습니다."
    if answer_mode not in {"easy", "detailed"}:
        raise ValueError(f"지원하지 않는 answer_mode: {answer_mode}")

    raw_context = format_raw_top5_for_llm(search_results)
    style_prompt = EASY_STYLE_PROMPT if answer_mode == "easy" else DETAILED_STYLE_PROMPT
    max_tokens = 1200 if answer_mode == "easy" else 1800
    user_prompt = f"""
    [사용자 질문]
    {_clean_text(question)}

    [Top 5 원본 청크 JSON]
    {raw_context}

    [출력 방식]
    {style_prompt}

    위 원본 청크만 사용하여 답변하세요.
    """.strip()

    try:
        response = HCX_CLIENT.chat.completions.create(
            model=HCX_CHAT_MODEL,
            messages=[
                {"role": "system", "content": COMMON_ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            max_tokens=max_tokens,
        )
    except BadRequestError as error:
        raise RuntimeError(
            "HCX가 Raw Top-5 요청을 거부했습니다. 선택 조건상 컨텍스트를 자르지 않으므로, "
            "Top 5 전체 길이가 모델 입력 한도를 넘었는지 원본 오류를 확인하세요. "
            f"원인={error}"
        ) from error

    answer = response.choices[0].message.content
    if not answer or not answer.strip():
        raise RuntimeError("HCX 답변 본문이 비어 있습니다.")
    answer = _remove_model_generated_urls(answer.strip())
    return _remove_invalid_chunk_citations(answer, len(search_results))


print("A안 Raw Top-5 answer generator 준비 완료")


In [ ]:
def _ordered_unique_strings(values: Iterable[Any]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for value in values:
        text = str(value or "").strip()
        if text and text not in seen:
            seen.add(text)
            ordered.append(text)
    return ordered


def build_rule_based_sources(search_results: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_url: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for result in search_results:
        chunk = result["chunk"]
        rank = int(result["rank"])
        for key, source_type in (
            ("source_url", "공식 페이지"),
            ("official_download_url", "공식 첨부파일"),
        ):
            url = str(chunk.get(key) or "").strip()
            if not url:
                continue
            row = by_url.setdefault(url, {
                "url": url,
                "source_type": source_type,
                "title": (
                    str(chunk.get("title") or "").strip()
                    or str(chunk.get("section_title") or "").strip()
                    or str(chunk.get("document_id") or "").strip()
                ),
                "chunk_citations": [],
            })
            citation = f"C{rank}"
            if citation not in row["chunk_citations"]:
                row["chunk_citations"].append(citation)
    return list(by_url.values())


def sources_to_markdown(sources: list[dict[str, Any]]) -> str:
    if not sources:
        return "### 출처\n\nTop 5 청크에서 공식 URL을 확인하지 못했습니다."
    lines = ["### 출처", ""]
    for source in sources:
        citations = ", ".join(source["chunk_citations"])
        title = str(source["title"] or "공식 출처").replace("[", "").replace("]", "")
        lines.append(
            f"- [{title}]({source['url']}) — {source['source_type']} ({citations})"
        )
    return "\n".join(lines)


def retrieval_table_markdown(results: list[dict[str, Any]]) -> str:
    lines = [
        "### 검색 결과",
        "",
        "|순위|청크 ID|Dense 순위|BM25 순위|Weighted RRF|제목 / 소제목|",
        "|---:|---|---:|---:|---:|---|",
    ]
    for result in results:
        chunk = result["chunk"]
        dense_rank = result["dense_rank"] if result["dense_rank"] is not None else "-"
        bm25_rank = result["bm25_rank"] if result["bm25_rank"] is not None else "-"
        title = " / ".join(
            part
            for part in [
                str(chunk.get("title") or "").replace("|", "\\|"),
                str(chunk.get("section_title") or "").replace("|", "\\|"),
            ]
            if part
        )
        lines.append(
            f"|{result['rank']}|{result['chunk_id']}|{dense_rank}|{bm25_rank}|"
            f"{result['rrf_score']:.6f}|{title}|"
        )
    return "\n".join(lines)


## 9. 질문 함수와 `자세한 답변 보기` 버튼

`ask()`는 검색을 한 번 수행하고 쉬운 답변을 생성합니다. `show_detailed_answer(result)`는 `result` 안에 저장된 **동일한 Top 5**를 재사용합니다. 따라서 버튼을 눌러도 재검색으로 근거가 바뀌지 않습니다.


In [ ]:
def ask(
    question: str,
    *,
    show_retrieval: bool = True,
    show_raw_top5: bool = False,
) -> dict[str, Any]:
    cleaned = _clean_text(question)
    if not cleaned:
        raise ValueError("질문이 비어 있습니다.")

    search_results = hybrid_search(cleaned)
    easy_answer = generate_raw_top5_answer(
        cleaned,
        search_results,
        answer_mode="easy",
    )
    sources = build_rule_based_sources(search_results)

    result = {
        "answer_method": "A_RAW_TOP5",
        "question": cleaned,
        "easy_answer": easy_answer,
        "detailed_answer": None,
        "sources": sources,
        "search_results": search_results,
        "raw_top5_text": format_raw_top5_for_llm(search_results),
    }

    display(Markdown(f"## 쉬운 답변\n\n{easy_answer}"))
    display(Markdown(sources_to_markdown(sources)))
    if show_retrieval:
        display(Markdown(retrieval_table_markdown(search_results)))
    if show_raw_top5:
        display(JSON(json.loads(result["raw_top5_text"]), expanded=False))
    return result


def show_detailed_answer(result: dict[str, Any]) -> str:
    if result.get("answer_method") != "A_RAW_TOP5":
        raise ValueError("A_RAW_TOP5 결과가 아닙니다.")
    if result.get("detailed_answer"):
        detailed_answer = str(result["detailed_answer"])
    else:
        detailed_answer = generate_raw_top5_answer(
            str(result["question"]),
            result["search_results"],
            answer_mode="detailed",
        )
        result["detailed_answer"] = detailed_answer

    display(Markdown(f"## 자세한 답변\n\n{detailed_answer}"))
    display(Markdown(sources_to_markdown(result["sources"])))
    return detailed_answer


def ask_with_detail_button(
    question: str,
    *,
    show_retrieval: bool = True,
    show_raw_top5: bool = False,
) -> dict[str, Any]:
    result = ask(
        question,
        show_retrieval=show_retrieval,
        show_raw_top5=show_raw_top5,
    )

    button = widgets.Button(
        description="자세한 답변 보기",
        button_style="primary",
        icon="book",
        tooltip="같은 Top 5 청크로 자세한 답변을 생성합니다.",
    )
    detail_output = widgets.Output()

    def on_click(_button: widgets.Button) -> None:
        _button.disabled = True
        _button.description = "자세한 답변 생성 중..."
        with detail_output:
            clear_output(wait=True)
            try:
                show_detailed_answer(result)
                _button.description = "자세한 답변 생성 완료"
                _button.icon = "check"
            except Exception as error:
                _button.disabled = False
                _button.description = "자세한 답변 다시 시도"
                _button.icon = "refresh"
                print(f"오류: {type(error).__name__}: {error}")

    button.on_click(on_click)
    display(button, detail_output)
    return result


print("ask(), show_detailed_answer(), ask_with_detail_button() 준비 완료")


def launch_chat_input(
    *,
    show_retrieval: bool = True,
    show_raw_top5: bool = False,
) -> dict[str, Any]:
    """Colab에서 반복 질문을 받을 수 있는 입력창과 전송 버튼을 표시합니다."""
    question_input = widgets.Textarea(
        value="",
        placeholder="예: 착오송금 반환지원 신청 대상과 신청 방법을 알려줘.",
        description="질문",
        disabled=False,
        layout=widgets.Layout(width="100%", height="92px"),
        style={"description_width": "55px"},
    )
    send_button = widgets.Button(
        description="질문 전송",
        button_style="success",
        icon="paper-plane",
        tooltip="입력한 질문으로 검색하고 쉬운 답변을 생성합니다.",
        layout=widgets.Layout(width="150px"),
    )
    clear_button = widgets.Button(
        description="대화 지우기",
        button_style="",
        icon="trash",
        tooltip="현재 출력 영역만 지웁니다.",
        layout=widgets.Layout(width="150px"),
    )
    status = widgets.HTML(value="질문을 입력한 뒤 <b>질문 전송</b>을 누르세요.")
    chat_output = widgets.Output(
        layout=widgets.Layout(
            width="100%",
            border="1px solid #d9d9d9",
            padding="12px",
            margin="10px 0 0 0",
        )
    )
    state: dict[str, Any] = {"results": []}

    def submit_question(_button: widgets.Button) -> None:
        question = _clean_text(question_input.value)
        if not question:
            status.value = '<span style="color:#c62828"><b>질문을 입력하세요.</b></span>'
            return

        send_button.disabled = True
        clear_button.disabled = True
        question_input.disabled = True
        status.value = '<span style="color:#1565c0"><b>검색하고 답변을 생성하는 중입니다...</b></span>'

        with chat_output:
            display(Markdown(f"---\n\n### 사용자 질문\n\n{question}"))
            try:
                result = ask_with_detail_button(
                    question,
                    show_retrieval=show_retrieval,
                    show_raw_top5=show_raw_top5,
                )
                state["results"].append(result)
                question_input.value = ""
                status.value = "답변 생성 완료. 다음 질문을 입력할 수 있습니다."
            except Exception as error:
                display(Markdown(
                    f"**오류:** `{type(error).__name__}` — {str(error)}"
                ))
                status.value = '<span style="color:#c62828"><b>오류가 발생했습니다. 메시지를 확인하세요.</b></span>'
            finally:
                send_button.disabled = False
                clear_button.disabled = False
                question_input.disabled = False

    def clear_chat(_button: widgets.Button) -> None:
        chat_output.clear_output()
        state["results"].clear()
        status.value = "출력 영역을 지웠습니다. 새 질문을 입력하세요."

    send_button.on_click(submit_question)
    clear_button.on_click(clear_chat)

    controls = widgets.HBox(
        [send_button, clear_button],
        layout=widgets.Layout(gap="8px"),
    )
    ui = widgets.VBox(
        [question_input, controls, status, chat_output],
        layout=widgets.Layout(width="100%"),
    )
    display(ui)
    return {
        "ui": ui,
        "question_input": question_input,
        "send_button": send_button,
        "clear_button": clear_button,
        "chat_output": chat_output,
        "state": state,
    }


print("Colab 질문 입력창 launch_chat_input() 준비 완료")


## 10. Colab 질문 입력창 실행

아래 셀을 실행하면 화면에 질문 입력창, `질문 전송`, `대화 지우기` 버튼이 표시됩니다.

- 질문을 입력하고 `질문 전송`을 누르면 쉬운 답변이 생성됩니다.
- 각 답변 아래 `자세한 답변 보기`를 누르면 같은 Top 5로 자세한 답변을 추가 생성합니다.
- 입력창은 한 번만 실행하면 여러 질문에 계속 사용할 수 있습니다.
- 런타임 재연결 후 입력창이 보이지 않으면 이 셀을 다시 실행하세요.


In [ ]:
chat = launch_chat_input(
    show_retrieval=True,
    show_raw_top5=False,
)


## 11. 입력창 없이 함수로 테스트하기

자동화된 비교 코드나 단일 테스트에서는 아래 함수를 직접 사용하면 됩니다. 자세한 답변은 검색을 다시 하지 않습니다.


In [ ]:
# result = ask("예금은 얼마까지 보호되나요?", show_retrieval=True)
# detailed_answer = show_detailed_answer(result)


## 12. Elasticsearch 상태 진단

검색 오류가 발생했을 때 아래 셀을 실행하면 서버, 플러그인, 인덱스, 문서 수, Nori 분석 결과를 한 번에 확인할 수 있습니다.


In [ ]:
def elasticsearch_diagnostics() -> dict[str, Any]:
    info = ES.info()
    nodes = ES.nodes.info(metric="plugins")
    plugins = sorted({
        str(plugin.get("name") or "")
        for node in nodes["nodes"].values()
        for plugin in node.get("plugins", [])
    })
    index_exists = bool(ES.indices.exists(index=ES_INDEX_NAME))
    document_count = int(ES.count(index=ES_INDEX_NAME)["count"]) if index_exists else None
    tokens = []
    if index_exists:
        tokens = [
            token["token"]
            for token in ES.indices.analyze(
                index=ES_INDEX_NAME,
                analyzer=ES_ANALYZER_NAME,
                text="착오송금 반환지원",
            )["tokens"]
        ]
    return {
        "connected": True,
        "url": ES_URL,
        "version": info["version"]["number"],
        "cluster_name": info["cluster_name"],
        "analysis_nori_installed": "analysis-nori" in plugins,
        "plugins": plugins,
        "index_name": ES_INDEX_NAME,
        "index_exists": index_exists,
        "document_count": document_count,
        "expected_document_count": len(CHUNKS),
        "nori_none_tokens": tokens,
    }


display(JSON(elasticsearch_diagnostics(), expanded=True))


## A안의 해석 범위

이 노트북은 검색 결과를 답변 모델에 직접 전달하는 최소 기준선입니다.

- 쉬운 답변과 자세한 답변이 같은 Top 5를 사용하도록 제어합니다.
- 그러나 두 답변 사이의 사실 일관성을 보장하는 공통 Fact Sheet는 없습니다.
- 검색 청크의 중복·충돌·대상 혼동도 별도 단계에서 해결하지 않습니다.
- 따라서 이후 B~E안과 비교할 때 A안의 품질과 호출 비용을 기준값으로 사용할 수 있습니다.
- 현재 노트북에는 평가 점수 산출, 평가 데이터셋 순회, 결과 저장은 포함하지 않았습니다.
